In [ ]:
include( "../model.jl" )
include( "../cluster.jl" )
include( "../prob.jl" )

using Optim
using ThreadsX

_, base = expfolder( "system" );

In [ ]:
# Density and speed parameters.
include( "size-0-params.jl" )

# System size.
N = 128

# Create folder and extract critical noise.
folderdata = [findfolder( N, ρlist[i], slist[j]; base=base, crit=true ) for i ∈ 1:Nρ, j ∈ 1:Ns]

# Extract cluster mass distribution exponent.
λdata = first.( readdlm.( folderdata.*"power-law.txt" ) )

# Import the cluster change frequency values.
θdatalist = readdlm.( folderdata.*"optimal-rates.txt" )

# Extract the indeces with data points.
# Note: Also enforcing upper bound.
ιdatalist = [[BitVector( [
    all( [isfinite( θdatalist[i,j][k1,k2] ) & (θdatalist[i,j][k1,k2] < 10.0) &
        (k1 < 3 ? 0.1 < θdatalist[i,j][k1,k2] : true) for k1 ∈ 1:3] )
    for k2 ∈ 1:N] ) for j ∈ 1:Ns] for i ∈ 1:Nρ];

In [ ]:
# Left and right distributions expectation.
λ1data = [mean( θdatalist[i,j][1,ιdatalist[i][j]] ) for i ∈ 1:Nρ, j ∈ 1:Ns]
λ2data = [mean( θdatalist[i,j][2,ιdatalist[i][j]] ) for i ∈ 1:Nρ, j ∈ 1:Ns]
κdata  = [mean( θdatalist[i,j][3,ιdatalist[i][j]] ) for i ∈ 1:Nρ, j ∈ 1:Ns];

In [ ]:
i = 6;  j = 6

# Plot the optimal rates for changing cluster size.
plt = plot(; layout=(3,1), size=(400,600), legend=false )

for k ∈ 1:2
    # Exponent for changing cluster size.
    scatter!( plt[k], 1:N, θdatalist[i,j][k,:]; color=:gray )

    # Exponent from the cluster mass distribution.
    plot!( plt[k], [1,N], fill( λdata[i,j], 2 ); color=:indianred, linestyle=:dash, lw=2 )

    # Plot details.
    plot!( plt[k]; xlims=(1,N), ylims=(0,3) )
end

# Add exected value.
plot!( plt[1], [1,N], fill( λ1data[i,j], 2 ); color=:cornflowerblue, linestyle=:dash, lw=2 )
plot!( plt[2], [1,N], fill( λ2data[i,j], 2 ); color=:cornflowerblue, linestyle=:dash, lw=2 )
plot!( plt[3], [1,N], fill(  κdata[i,j], 2 ); color=:cornflowerblue, linestyle=:dash, lw=2 )

scatter!( plt[3], 1:N, θdatalist[i,j][3,:]; color=:gray )
plot!( plt[3]; xlims=(1,N), ylims=(0,1) )

In [ ]:
# Compile the approximated rate variables.
θ̂datalist = [[fill( λ1data[i,j], N ) fill( λ2data[i,j], N ) fill( κdata[i,j], N )] for i ∈ 1:Nρ, j ∈ 1:Ns];

In [ ]:
if true
    # Save the cluster change parameters for each parameter case.
    for i ∈ 1:Nρ, j ∈ 1:Ns
        writedlm( folderdata[i,j]*"optimal-rates-hat.txt", θ̂datalist[i,j] )
    end
end